# UNet++: Two-Stage Training (One Notebook)

Two-stage training on trainable_pool with instance metrics and final visualization.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('.').resolve()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
ACTIVE_PYTHON = Path(sys.executable)

print('Project root:', PROJECT_ROOT)
print('Active kernel python:', ACTIVE_PYTHON)
print('Venv python:', VENV_PYTHON if VENV_PYTHON.exists() else 'not found')

if not VENV_PYTHON.exists():
    print(f"[warn] Python from .venv not found: {VENV_PYTHON}. Will use active kernel python where possible.")


In [ ]:
# Paths and logs
SPLITS_DIR = Path('data/splits')
LOG_STAGE1 = Path('runs/unetpp_stage1_train.log')
LOG_STAGE2 = Path('runs/unetpp_stage2_train.log')
RUN_DIR = Path('runs/unetpp_two_stage')
RUN_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_BEST = RUN_DIR / 'stage1_best.pt'
STAGE2_BEST = RUN_DIR / 'stage2_best.pt'


In [ ]:
# Helper to run shell commands with notebook-friendly progress + realtime curves
import re
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt


def run_cmd(
    cmd,
    cwd='.',
    log_path=None,
    epoch_total=None,
    quiet_tqdm_lines=True,
    live_plots=True,
    live_plot_every=1,
):
    print('\n>>>', ' '.join(cmd))

    log_f = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_f = log_path.open('w', encoding='utf-8')
        print('logging to:', log_path)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'

    train_pbar = None
    val_pbar = None
    current_epoch = None

    hist_train_loss = []
    hist_val_f1 = []
    plot_handle = None

    train_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[train\]:.*?\|\s*(\d+)/(\d+)\s*\[.*loss=([0-9.]+)")
    val_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[val\]:.*?\|\s*(\d+)/(\d+)\s*\[")
    epoch_summary_re = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    def redraw_curves(final=False):
        nonlocal plot_handle
        if not live_plots or not hist_train_loss:
            return
        if (not final) and (len(hist_train_loss) % max(1, int(live_plot_every)) != 0):
            return

        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(hist_train_loss, label='train_loss')
        ax[0].set_title('Train Loss')
        ax[0].legend()

        ax[1].plot(hist_val_f1, label='val_f1')
        ax[1].set_title('Val F1')
        ax[1].legend()

        if plot_handle is None:
            plot_handle = display(fig, display_id=True)
        else:
            plot_handle.update(fig)
        plt.close(fig)

    def ensure_epoch(epoch_num):
        nonlocal current_epoch, train_pbar, val_pbar
        if current_epoch == epoch_num:
            return

        if train_pbar is not None:
            train_pbar.close()
            train_pbar = None
        if val_pbar is not None:
            val_pbar.close()
            val_pbar = None

        current_epoch = epoch_num

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(Path(cwd).resolve()),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )

        for raw in proc.stdout:
            if log_f is not None:
                log_f.write(raw)

            line = raw.rstrip('\n')

            mt = train_re.search(line)
            if mt:
                ep = int(mt.group(1)); ep_tot = int(mt.group(2))
                cur = int(mt.group(3)); tot = int(mt.group(4)); loss = float(mt.group(5))
                ensure_epoch(ep)
                if train_pbar is None or train_pbar.total != tot:
                    if train_pbar is not None:
                        train_pbar.close()
                    train_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [train]')
                if cur >= train_pbar.n:
                    train_pbar.update(cur - train_pbar.n)
                train_pbar.set_postfix(loss=f'{loss:.4f}')
                continue

            mv = val_re.search(line)
            if mv:
                ep = int(mv.group(1)); ep_tot = int(mv.group(2))
                cur = int(mv.group(3)); tot = int(mv.group(4))
                ensure_epoch(ep)
                if val_pbar is None or val_pbar.total != tot:
                    if val_pbar is not None:
                        val_pbar.close()
                    val_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [val]')
                if cur >= val_pbar.n:
                    val_pbar.update(cur - val_pbar.n)
                continue

            me = epoch_summary_re.search(line)
            if me:
                tl = float(me.group(3)); vf1 = float(me.group(4))
                hist_train_loss.append(tl)
                hist_val_f1.append(vf1)
                redraw_curves(final=False)
                print(line)
                continue

            if line.startswith('Epoch time:'):
                print(line)
                continue

            if 'Saved best:' in line or 'Loaded init checkpoint:' in line:
                print(line)
            elif not quiet_tqdm_lines and line:
                print(line)

        proc.wait()
    finally:
        redraw_curves(final=True)
        if train_pbar is not None:
            train_pbar.close()
        if val_pbar is not None:
            val_pbar.close()
        if log_f is not None:
            log_f.close()

    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}')




In [ ]:
# Step 1: generate/re-generate two-stage ID splits
run_cmd([
    str(VENV_PYTHON),
    'tools/make_two_stage_ids.py',
    '--images_dir', 'trainable_pool/images',
    '--instances_dir', 'trainable_pool/instances',
    '--out_dir', str(SPLITS_DIR),
    '--cups_prefix', 'data_cups',
    '--val_split_stage1', '0.15',
    '--val_split_stage2', '0.15',
    '--seed', '42',
])


In [ ]:
# Show split sizes
for p in [
    SPLITS_DIR / 'stage1_pretrain_train_ids.txt',
    SPLITS_DIR / 'stage1_pretrain_val_ids.txt',
    SPLITS_DIR / 'stage2_finetune_train_ids.txt',
    SPLITS_DIR / 'stage2_finetune_val_ids.txt',
]:
    n = len([x for x in p.read_text(encoding='utf-8').splitlines() if x.strip()])
    print(f'{p}: {n}')


In [ ]:
# Step 2: Stage 1 training (UNet++ pretrain on non-cups)
# Ensure dependencies are installed in active notebook kernel (important for nbconvert runs).
import importlib.util

_required = {
    'cv2': 'opencv-python',
    'skimage': 'scikit-image',
    'segmentation_models_pytorch': 'segmentation-models-pytorch',
    'timm': 'timm',
}
_missing = [pip_name for mod_name, pip_name in _required.items() if importlib.util.find_spec(mod_name) is None]
if _missing:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-U', *_missing])

# Keep .venv in sync when it exists (used by helper scripts in this notebook).
if VENV_PYTHON.exists() and Path(VENV_PYTHON) != Path(sys.executable):
    run_cmd([str(VENV_PYTHON), '-m', 'pip', 'install', '-U', 'opencv-python', 'segmentation-models-pytorch', 'timm', 'scikit-image'])

import cv2, numpy as np, torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from colonyseg.metrics.instance_metrics import instance_scores

IMG_SIZE=512; BS=4; E1=35; E2=35
device='cuda' if torch.cuda.is_available() else 'cpu'
print('device', device)

def _ids(p): return [x.strip() for x in Path(p).read_text(encoding='utf-8').splitlines() if x.strip()]
def _find(image_id):
    for ext in ('.png','.jpg','.jpeg','.tif','.tiff','.bmp'):
        p=Path('trainable_pool/images')/f'{image_id}{ext}'
        if p.exists(): return p
    raise FileNotFoundError(image_id)

class DS(Dataset):
    def __init__(self, ids): self.ids=ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        image_id=self.ids[i]
        img=cv2.imread(str(_find(image_id)), cv2.IMREAD_COLOR)
        inst=cv2.imread(str(Path('trainable_pool/instances')/f'{image_id}.png'), cv2.IMREAD_UNCHANGED)
        img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_AREA)
        inst=cv2.resize(inst.astype(np.int32),(IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        sem=(inst>0).astype(np.float32)
        x=torch.from_numpy(img.transpose(2,0,1)).float()/255.
        y=torch.from_numpy(sem).unsqueeze(0).float()
        return {'x':x,'y':y,'inst':torch.from_numpy(inst.astype(np.int32))}

def _dice_loss(logits,y,eps=1e-6):
    p=torch.sigmoid(logits); inter=(p*y).sum((2,3)); union=p.sum((2,3))+y.sum((2,3)); return 1.-((2*inter+eps)/(union+eps)).mean()
def _sem2inst(sem,t=0.5,min_d=3,area_min=4):
    b=(sem>=t).astype(np.uint8)
    if b.sum()==0: return np.zeros_like(b,dtype=np.int32)
    d=ndi.distance_transform_edt(b)
    c=peak_local_max(d, labels=b, min_distance=min_d)
    m=np.zeros_like(b,dtype=np.int32)
    for i,(y,x) in enumerate(c,1): m[y,x]=i
    if m.max()==0: return np.zeros_like(b,dtype=np.int32)
    lab=watershed(-d,m,mask=b.astype(bool)).astype(np.int32)
    out=lab.copy()
    for v in np.unique(lab):
        if v!=0 and (lab==v).sum()<area_min: out[lab==v]=0
    u=np.unique(out); u=u[u!=0]
    z=np.zeros_like(out,dtype=np.int32)
    for i,v in enumerate(u,1): z[out==v]=i
    return z

def _train(train_ids,val_ids,epochs,lr,init_ckpt=None,save_ckpt=None,log_path=None):
    model=smp.UnetPlusPlus(encoder_name='timm-efficientnet-b3', encoder_weights='imagenet', in_channels=3, classes=1, activation=None).to(device)
    if init_ckpt and Path(init_ckpt).exists(): model.load_state_dict(torch.load(init_ckpt,map_location='cpu')['model'])
    tr=DataLoader(DS(train_ids),batch_size=BS,shuffle=True,num_workers=0,pin_memory=True,drop_last=True)
    va=DataLoader(DS(val_ids),batch_size=1,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best=-1
    lf=Path(log_path).open('w',encoding='utf-8')
    for e in range(1,epochs+1):
        model.train(); ls=0.; n=0
        for b in tr:
            x=b['x'].to(device,non_blocking=True); y=b['y'].to(device,non_blocking=True)
            logits=model(x); loss=F.binary_cross_entropy_with_logits(logits,y)+_dice_loss(logits,y)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            ls+=float(loss.item()); n+=1
        tl=ls/max(1,n)
        model.eval(); mets=[]
        with torch.no_grad():
            for b in va:
                x=b['x'].to(device,non_blocking=True); gt=b['inst'][0].numpy().astype(np.int32)
                sem=torch.sigmoid(model(x))[0,0].cpu().numpy(); pr=_sem2inst(sem)
                gt=cv2.resize(gt,(pr.shape[1],pr.shape[0]),interpolation=cv2.INTER_NEAREST)
                mets.append(instance_scores(gt,pr,0.5))
        f1=float(np.mean([m['f1'] for m in mets])) if mets else 0.
        me=float(np.mean([m['merge'] for m in mets])) if mets else 0.
        sp=float(np.mean([m['split'] for m in mets])) if mets else 0.
        ce=float(np.mean([m['count_err'] for m in mets])) if mets else 0.
        if f1>best: best=f1; torch.save({'model':model.state_dict(),'best_f1':best}, save_ckpt)
        line=f'Epoch {e}/{epochs} | train_loss={tl:.4f} | val_f1={f1:.4f} | merge={me:.3f} | split={sp:.3f} | count_err={ce:.3f}'
        print(line); lf.write(line+'\n')
    lf.close(); return model

s1_tr=_ids(SPLITS_DIR/'stage1_pretrain_train_ids.txt'); s1_va=_ids(SPLITS_DIR/'stage1_pretrain_val_ids.txt')
_train(s1_tr,s1_va,E1,3e-4,None,STAGE1_BEST,LOG_STAGE1)


In [ ]:
# Verify Stage 1 checkpoint exists
print('stage1 exists:', STAGE1_BEST.exists(), STAGE1_BEST)
if not STAGE1_BEST.exists():
    raise FileNotFoundError(STAGE1_BEST)


In [ ]:
# Step 3: Stage 2 training (finetune on cups only)
s2_tr=_ids(SPLITS_DIR/'stage2_finetune_train_ids.txt'); s2_va=_ids(SPLITS_DIR/'stage2_finetune_val_ids.txt')
_train(s2_tr,s2_va,E2,1.5e-4,STAGE1_BEST,STAGE2_BEST,LOG_STAGE2)


In [ ]:
# Verify Stage 2 checkpoints
print('stage2 exists:', STAGE2_BEST.exists(), STAGE2_BEST)
if not STAGE2_BEST.exists():
    raise FileNotFoundError(STAGE2_BEST)


In [ ]:
# Training curves from logs
import re
import numpy as np
import matplotlib.pyplot as plt


def parse_train_log(path):
    path = Path(path)
    if not path.exists():
        print(f'log missing: {path}')
        return None

    epoch, train_loss = [], []
    val_f1, val_merge, val_split, val_count = [], [], [], []

    pat = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    for ln in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = pat.search(ln)
        if not m:
            continue
        epoch.append(int(m.group(1)))
        train_loss.append(float(m.group(3)))
        val_f1.append(float(m.group(4)))
        val_merge.append(float(m.group(5)))
        val_split.append(float(m.group(6)))
        val_count.append(float(m.group(7)))

    if not epoch:
        print(f'no parsed epoch lines in: {path}')
        return None

    return {
        'epoch': np.array(epoch),
        'train_loss': np.array(train_loss),
        'val_f1': np.array(val_f1),
        'val_merge': np.array(val_merge),
        'val_split': np.array(val_split),
        'val_count_err': np.array(val_count),
        'path': path,
    }


def plot_stage_curves(data, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(data['train_loss'], label='train_loss')
    ax[0].set_title('Train Loss')
    ax[0].legend()

    ax[1].plot(data['val_f1'], label='val_f1')
    ax[1].set_title('Val F1')
    ax[1].legend()

    print(title)
    plt.show()


stage1 = parse_train_log(LOG_STAGE1)
stage2 = parse_train_log(LOG_STAGE2)

if stage1 is not None:
    print('Stage1 epochs:', len(stage1['epoch']), 'best val_f1:', float(stage1['val_f1'].max()))
    plot_stage_curves(stage1, 'Stage 1: Pretrain (non-cups)')

if stage2 is not None:
    print('Stage2 epochs:', len(stage2['epoch']), 'best val_f1:', float(stage2['val_f1'].max()))
    plot_stage_curves(stage2, 'Stage 2: Finetune (cups)')



## Notes

- UNet++ predicts semantic map, instances are recovered by watershed.
- Tune `t_sem`, `min_distance`, `area_min` for better instance counts.


In [ ]:
# Final visualization cell
import cv2, torch, numpy as np, matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

model=smp.UnetPlusPlus(encoder_name='timm-efficientnet-b3', encoder_weights='imagenet', in_channels=3, classes=1, activation=None).to(device)
model.load_state_dict(torch.load(STAGE2_BEST,map_location='cpu')['model'])
model.eval()
img=cv2.imread('IMG_4377.jpg', cv2.IMREAD_COLOR)
img_rgb=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
x=cv2.resize(img_rgb,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_AREA)
x_t=torch.from_numpy(x.transpose(2,0,1)).float().unsqueeze(0).to(device)/255.
with torch.no_grad(): sem=torch.sigmoid(model(x_t))[0,0].cpu().numpy()
pr=_sem2inst(sem)
pr=cv2.resize(pr.astype(np.int32),(img_rgb.shape[1],img_rgb.shape[0]),interpolation=cv2.INTER_NEAREST)
edges=cv2.Canny((pr>0).astype(np.uint8)*255,50,150)>0
vis=img_rgb.copy(); vis[edges]=[0,255,0]
plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.title('Input'); plt.imshow(img_rgb); plt.axis('off')
plt.subplot(1,2,2); plt.title(f'UNet++ instances: {int(np.max(pr))}'); plt.imshow(vis); plt.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# === MLFLOW AUTO LOGGING ===
import os
import re
import json
from pathlib import Path

try:
    import mlflow
except Exception as exc:
    print(f"[warn] mlflow is unavailable: {exc}")
else:
    tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000')
    experiment = os.getenv('MLFLOW_EXPERIMENT', 'colony_models')
    run_name = os.getenv('MLFLOW_RUN_NAME', 'unetpp_two_stage_notebook')

    def _parse_train_log(path):
        p = Path(path)
        if not p.exists():
            return None
        pat = re.compile(
            r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
        )
        out = {
            'epoch': [],
            'train_loss': [],
            'val_f1': [],
            'val_merge': [],
            'val_split': [],
            'val_count_err': [],
        }
        for ln in p.read_text(encoding='utf-8', errors='ignore').splitlines():
            m = pat.search(ln)
            if not m:
                continue
            out['epoch'].append(int(m.group(1)))
            out['train_loss'].append(float(m.group(3)))
            out['val_f1'].append(float(m.group(4)))
            out['val_merge'].append(float(m.group(5)))
            out['val_split'].append(float(m.group(6)))
            out['val_count_err'].append(float(m.group(7)))
        return out if out['epoch'] else None

    s1 = _parse_train_log(LOG_STAGE1)
    s2 = _parse_train_log(LOG_STAGE2)

    summary = {
        'stage1': {
            'log': str(LOG_STAGE1),
            'best_ckpt': str(STAGE1_BEST),
            'epochs_logged': len(s1['epoch']) if s1 else 0,
            'best_val_f1': (max(s1['val_f1']) if s1 and s1['val_f1'] else None),
        },
        'stage2': {
            'log': str(LOG_STAGE2),
            'best_ckpt': str(STAGE2_BEST),
            'epochs_logged': len(s2['epoch']) if s2 else 0,
            'best_val_f1': (max(s2['val_f1']) if s2 and s2['val_f1'] else None),
        },
    }
    summary_path = RUN_DIR / 'mlflow_summary_unetpp_two_stage.json'
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment)
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({
            'pipeline': 'unetpp_two_stage_notebook',
            'model_name': 'UnetPlusPlus',
        })

        mlflow.log_param('run_dir', str(RUN_DIR))
        mlflow.log_param('splits_dir', str(SPLITS_DIR))
        mlflow.log_param('img_size', int(globals().get('IMG_SIZE', 512)))
        if 'E1' in globals():
            mlflow.log_param('stage1_epochs_cfg', int(E1))
        if 'E2' in globals():
            mlflow.log_param('stage2_epochs_cfg', int(E2))

        if s1:
            for step, val in enumerate(s1['train_loss'], start=1):
                mlflow.log_metric('stage1_train_loss', float(val), step=step)
            for step, val in enumerate(s1['val_f1'], start=1):
                mlflow.log_metric('stage1_val_f1', float(val), step=step)
            for step, val in enumerate(s1['val_merge'], start=1):
                mlflow.log_metric('stage1_val_merge', float(val), step=step)
            for step, val in enumerate(s1['val_split'], start=1):
                mlflow.log_metric('stage1_val_split', float(val), step=step)
            for step, val in enumerate(s1['val_count_err'], start=1):
                mlflow.log_metric('stage1_val_count_err', float(val), step=step)
            mlflow.log_metric('stage1_best_val_f1', float(max(s1['val_f1'])), step=len(s1['val_f1']))

        if s2:
            for step, val in enumerate(s2['train_loss'], start=1):
                mlflow.log_metric('stage2_train_loss', float(val), step=step)
            for step, val in enumerate(s2['val_f1'], start=1):
                mlflow.log_metric('stage2_val_f1', float(val), step=step)
            for step, val in enumerate(s2['val_merge'], start=1):
                mlflow.log_metric('stage2_val_merge', float(val), step=step)
            for step, val in enumerate(s2['val_split'], start=1):
                mlflow.log_metric('stage2_val_split', float(val), step=step)
            for step, val in enumerate(s2['val_count_err'], start=1):
                mlflow.log_metric('stage2_val_count_err', float(val), step=step)
            mlflow.log_metric('stage2_best_val_f1', float(max(s2['val_f1'])), step=len(s2['val_f1']))

        for p in [LOG_STAGE1, LOG_STAGE2, STAGE1_BEST, STAGE2_BEST, summary_path]:
            pp = Path(p)
            if pp.exists():
                mlflow.log_artifact(str(pp), artifact_path='notebook_unetpp_two_stage')

    print('MLflow logging complete:', run_name)
